In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
df = pd.read_csv('data/processed/dados_projeto_limpos.csv')

In [ ]:
## Pra criar uma função que filtre os Top N de alguma métrica, sem perder o histórico #de release_year, que é essencial para a análise de evolução temporal. Assim, podemos #focar nos modelos mais relevantes sem perder a visão do panorama geral.
#def top_n_history(df, y_col, x_col, group_col=None, n=10):
#    # Ordena o DataFrame por uma coluna de valor de interesse e pega os Top N:
#    top_n = df.sort_values(by=y_col, ascending=False).head(n)
#
#    #Filtra o dataframe original para manter o histórico de release_year dos Top N e #posteriormente agrupa os dados por outra coluna desejada (como por exemplo: #model_slug, provider, etc.) para facilitar a análise de evolução temporal pra cada #campo analisável:
#    if group_col:
#        filtered_df = df[df[group_col].isin(top_n[group_col])].groupby([x_col, #group_col])[y_col].mean().reset_index()
#    else:
#        filtered_df = df[df['model_slug'].isin(top_n['model_slug'])].groupby(x_col)#[y_col].mean().reset_index()
#
#
#    return filtered_df

def top_n_history(df, x_col, y_col, group_col=None, n=10):
    #df é o DataFrame original, y_col é a coluna de valor que queremos analisar, x_col é a coluna do eixo x, group_col é a coluna pela qual queremos agrupar (como model_slug ou provider) e n é o número de top que queremos manter.
    if not group_col:
        # Se group_col não for fornecido, não agrupamos e apenas pegamos os top n de y_col
        top_n = df.sort_values(by=y_col, ascending=False).head(n)
        return top_n[[x_col, y_col]]
    top_groups = df.groupby(group_col)[y_col].mean().nlargest(n).index
    filtered_df = df[df[group_col].isin(top_groups)]
    return filtered_df.groupby([x_col, group_col])[y_col].mean().reset_index()

#exemplo de uso:
#top_n_df = top_n_history(df, x_col='release_year', y_col='blended_cost_usd_per_1m', group_col='model_slug', n=10)
#ou
#top_n_df = top_n_history(df, x_col='release_year', y_col='blended_cost_usd_per_1m', group_col='provider', n=10)
#ou ainda 
#top_n_df = top_n_history(df, x_col='release_year', y_col='blended_cost_usd_per_1m', group_col='model_family', n=10)


In [ ]:
#criar uma função de plotagem de linha com a biblioteca seaborn:
def plot_performance_line(data, x_col, y_col, title):

    """
    Plota um gráfico de linha para desempenho ao longo do tempo.

    Args:
        data (pd.DataFrame): DataFrame contendo os dados a serem plotados.
        x_col (str): Nome da coluna para o eixo x (geralmente data).
        y_col (str): Nome da coluna para o eixo y (geralmente métrica de desempenho).
        title (str): Título do gráfico.

    Returns:
        matplotlib.figure.Figure: Objeto da figura do gráfico.
    """
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=data, x=x_col, y=y_col, marker='o')
    plt.title(title)
    plt.xlabel(x_col.capitalize())
    plt.ylabel(y_col.capitalize())
    plt.xticks(rotation=45)
    plt.tight_layout()
    return plt.gcf()

In [ ]:
def plot_performance_line(data, x_col, y_col, title=None, group_col=None, n=None):
    sns.set_theme(style="whitegrid")
    # Se n for fornecido, filtra os top n antes de plotar
    if n and group_col:
        top_groups = data.groupby(group_col)[y_col].mean().nlargest(n).index
        data = data[data[group_col].isin(top_groups)]
        data = data.groupby([x_col, group_col])[y_col].mean().reset_index()
    elif n:
        top_n = data.sort_values(by=y_col, ascending=False).head(n)
        data = top_n[[x_col, y_col]]
    

    plt.figure(figsize=(12, 6))
    
    # O segredo está aqui: o parâmetro hue recebe a coluna de categorias para diferenciar as linhas por cor
    sns.lineplot(data=data, x=x_col, y=y_col, hue=group_col, marker='o')
    
    plt.title(title)
    plt.xlabel(x_col.replace('_', ' ').capitalize())
    plt.ylabel(y_col.replace('_', ' ').capitalize())
    plt.xticks(rotation=45)
    plt.grid(True, which='both', linestyle='--', alpha=0.7)
    plt.legend(title=group_col.replace('_', ' ').capitalize())
    sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()

    return plt.gcf()

#Exemplo de uso:
#fig_line = plot_performance_line(data, 'release_year', 'intelligence_per_dollar', 'Inteligência por Dólar no tempo', group_col='model_slug', n=10)

In [ ]:
#criar uma função de plotagem de barras com a biblioteca seaborn:
def plot_performance_bar(data, x_col, y_col, title):
    """
    Plota um gráfico de barras para desempenho por categoria.

    Args:
        data (pd.DataFrame): DataFrame contendo os dados a serem plotados.
        x_col (str): Nome da coluna para o eixo x (geralmente categoria).
        y_col (str): Nome da coluna para o eixo y (geralmente métrica de desempenho).
        title (str): Título do gráfico.

    Returns:
        matplotlib.figure.Figure: Objeto da figura do gráfico.
    """
    plt.figure(figsize=(10, 6))
    sns.barplot(data=data, x=x_col, y=y_col)
    plt.title(title)
    plt.xlabel(x_col.capitalize())
    plt.ylabel(y_col.capitalize())
    plt.xticks(rotation=45)
    plt.tight_layout()
    return plt.gcf()

In [ ]:
#criar uma função de plotagem de dispersão com a biblioteca seaborn:
def plot_performance_scatter(data, x_col, y_col, title):
    """
    Plota um gráfico de dispersão para desempenho por categoria.

    Args:
        data (pd.DataFrame): DataFrame contendo os dados a serem plotados.
        x_col (str): Nome da coluna para o eixo x (geralmente categoria).
        y_col (str): Nome da coluna para o eixo y (geralmente métrica de desempenho).
        title (str): Título do gráfico.

    Returns:
        matplotlib.figure.Figure: Objeto da figura do gráfico.
    """
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=data, x=x_col, y=y_col)
    plt.title(title)
    plt.xlabel(x_col.capitalize())
    plt.ylabel(y_col.capitalize())
    plt.xticks(rotation=45)
    plt.tight_layout()
    return plt.gcf()


In [ ]:
#criar uma função de plotagem de área com a biblioteca seaborn:
def plot_performance_area(data, x_col, y_col, title):
    """
    Plota um gráfico de área para desempenho ao longo do tempo.

    Args:
        data (pd.DataFrame): DataFrame contendo os dados a serem plotados.
        x_col (str): Nome da coluna para o eixo x (geralmente data).
        y_col (str): Nome da coluna para o eixo y (geralmente métrica de desempenho).
        title (str): Título do gráfico.

    Returns:
        matplotlib.figure.Figure: Objeto da figura do gráfico.
    """
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=data, x=x_col, y=y_col, marker='o')
    plt.fill_between(data[x_col], data[y_col], alpha=0.3)
    plt.title(title)
    plt.xlabel(x_col.capitalize())
    plt.ylabel(y_col.capitalize())
    plt.xticks(rotation=45)
    plt.tight_layout()
    return plt.gcf()

In [ ]:
#criar uma função de plotagem de rosca com a biblioteca seaborn:
def plot_performance_donut(data, x_col, y_col, title):
    """
    Plota um gráfico de rosca para desempenho por categoria.

    Args:
        data (pd.DataFrame): DataFrame contendo os dados a serem plotados.
        x_col (str): Nome da coluna para as categorias.
        y_col (str): Nome da coluna para os valores.
        title (str): Título do gráfico.

    Returns:
        matplotlib.figure.Figure: Objeto da figura do gráfico.
    """
    plt.figure(figsize=(8, 8))
    plt.pie(data[y_col], labels=data[x_col], autopct='%1.1f%%', startangle=140)
    plt.title(title)
    centre_circle = plt.Circle((0, 0), 0.70, fc='white')
    fig = plt.gcf()
    fig.gca().add_artist(centre_circle)
    plt.tight_layout()
    return fig

In [ ]:
line_price_performance = plot_performance_line(df, 'release_year', 'price_performance_ratio', title=None, group_col='generic_model_name', n=10)







In [ ]:
line_intelligence = plot_performance_line(df, 'release_year', 'aa_intelligence_index', title=None, group_col='generic_model_name', n=10)

In [ ]:
line_speed_dollar = plot_performance_line(df, 'release_year', 'speed_per_dollar', title=None, group_col='generic_model_name', n=10)